# Modular Data Sanitization & Exploration Engine

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler, OneHotEncoder, OrdinalEncoder
from google.colab import files
import io
import warnings
warnings.filterwarnings('ignore')
from IPython.display import display, HTML

## 2. PlottingMethods Class

In [ ]:
class PlottingMethods:
    """Helper class for granular Plotly visualizations."""
    
    @staticmethod
    def plot_bar(data: pd.Series, title: str = "Bar Chart") -> str:
        """Generate bar chart with counts and percentages."""
        counts = data.value_counts()
        percentages = (counts / counts.sum() * 100).round(2)
        
        fig = px.bar(
            x=counts.index, 
            y=counts.values,
            labels={'x': data.name or 'Category', 'y': 'Count'},
            title=title,
            text=percentages.astype(str) + '%'
        )
        fig.update_traces(textposition='outside')
        fig.update_layout(height=500)
        return fig.to_html(full_html=False, include_plotlyjs='cdn')
    
    @staticmethod
    def plot_pie(data: pd.Series, title: str = "Pie Chart") -> str:
        """Generate interactive pie chart."""
        fig = px.pie(
            names=data.value_counts().index, 
            values=data.value_counts().values,
            title=title
        )
        fig.update_layout(height=500)
        return fig.to_html(full_html=False, include_plotlyjs='cdn')
    
    @staticmethod
    def plot_histogram(data: pd.Series, title: str = "Histogram") -> str:
        """Generate histogram with box marginal."""
        fig = px.histogram(
            data, 
            nbins=30, 
            title=title,
            marginal="box",
            opacity=0.75
        )
        fig.update_layout(height=500)
        return fig.to_html(full_html=False, include_plotlyjs='cdn')

## 3. DataInspector Class

In [ ]:
class DataInspector:
    """Main class for data ingestion, cleaning, exploration and visualization."""
    
    def __init__(self):
        self.df = None
        self.original_df = None
        self.plotter = PlottingMethods()
    
    def upload_data(self):
        """Upload CSV file in Google Colab."""
        print("📤 Please upload your CSV file...")
        uploaded = files.upload()
        
        if not uploaded:
            print("❌ No file uploaded.")
            return False
        
        filename = list(uploaded.keys())[0]
        try:
            self.df = pd.read_csv(io.BytesIO(uploaded[filename]))
            self.original_df = self.df.copy()
            print(f"✅ Successfully loaded '{filename}' | Shape: {self.df.shape}")
            self._handle_garbage_strings()
            self.auto_correct_types()
            return True
        except Exception as e:
            print(f"❌ Error loading file: {e}")
            return False
    
    def _handle_garbage_strings(self):
        """Convert common garbage strings to NaN."""
        garbage = ['?', 'n/a', 'N/A', 'null', 'NULL', 'NaN', 'nan', ' ', '', '-']
        self.df = self.df.replace(garbage, np.nan)
    
    def auto_correct_types(self):
        """Force numeric conversion where possible."""
        for col in self.df.columns:
            if self.df[col].dtype == 'object':
                converted = pd.to_numeric(self.df[col], errors='coerce')
                if converted.notna().sum() > 0:
                    self.df[col] = converted
                    print(f"🔄 Auto-converted '{col}' to numeric.")
    
    def get_summary(self):
        """Display comprehensive data summary."""
        if self.df is None:
            print("No data loaded.")
            return
        
        print("="*70)
        print("📊 DATA SUMMARY")
        print("="*70)
        print(f"Rows: {self.df.shape[0]:,}")
        print(f"Columns: {self.df.shape[1]}")
        
        num_cols = self.df.select_dtypes(include=[np.number]).columns.tolist()
        cat_cols = self.df.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()
        
        print(f"Numerical columns: {len(num_cols)}")
        print(f"Categorical columns: {len(cat_cols)}")
        print("\nFirst 20 rows:")
        display(self.df.head(20))
        
        missing = self.df.isnull().sum()
        if missing.sum() > 0:
            print("\nMissing Values:")
            display(missing[missing > 0])
    
    def handle_missing_values(self, strategy: str = 'mean', constant: float = 0, columns: list = None):
        """Handle missing values with multiple strategies."""
        if self.df is None:
            print("No data loaded.")
            return
        
        target_cols = columns if columns else self.df.columns
        
        for col in target_cols:
            if col not in self.df.columns or self.df[col].isnull().sum() == 0:
                continue
                
            if strategy == 'mean' and pd.api.types.is_numeric_dtype(self.df[col]):
                fill_val = self.df[col].mean()
            elif strategy == 'median' and pd.api.types.is_numeric_dtype(self.df[col]):
                fill_val = self.df[col].median()
            elif strategy == 'mode':
                fill_val = self.df[col].mode()[0] if not self.df[col].mode().empty else np.nan
            elif strategy == 'constant':
                fill_val = constant
            else:
                continue
                
            self.df[col] = self.df[col].fillna(fill_val)
        
        print(f"✅ Missing values handled using '{strategy}' strategy.")
    
    def remove_duplicates(self):
        """Remove exact duplicate rows."""
        before = len(self.df)
        self.df = self.df.drop_duplicates().reset_index(drop=True)
        removed = before - len(self.df)
        print(f"✅ Removed {removed} duplicate rows.")
    
    def handle_outliers(self, columns: list = None, method: str = 'cap', factor: float = 1.5):
        """IQR-based outlier handling (cap or remove)."""
        if columns is None:
            columns = self.df.select_dtypes(include=[np.number]).columns.tolist()
        
        for col in columns:
            if col not in self.df.columns or not pd.api.types.is_numeric_dtype(self.df[col]):
                continue
                
            Q1 = self.df[col].quantile(0.25)
            Q3 = self.df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - factor * IQR
            upper = Q3 + factor * IQR
            
            if method == 'cap':
                self.df[col] = self.df[col].clip(lower=lower, upper=upper)
            elif method == 'remove':
                self.df = self.df[(self.df[col] >= lower) & (self.df[col] <= upper)]
        
        print(f"✅ Outliers handled ({method}) on {len(columns)} columns.")
    
    def delete_rows(self):
        """Interactive row deletion."""
        print(f"Current index range: {self.df.index.min()} to {self.df.index.max()}")
        inp = input("Enter row indices to delete (comma-separated): ")
        if not inp.strip():
            return
        try:
            indices = [int(x.strip()) for x in inp.split(',')]
            self.df = self.df.drop(indices, errors='ignore').reset_index(drop=True)
            print("✅ Rows deleted.")
        except:
            print("❌ Invalid input.")
    
    def delete_columns(self):
        """Interactive column deletion."""
        print("Available columns:", list(self.df.columns))
        inp = input("Enter columns to delete (comma-separated): ")
        if not inp.strip():
            return
        cols = [x.strip() for x in inp.split(',')]
        self.df = self.df.drop(columns=[c for c in cols if c in self.df.columns])
        print("✅ Columns deleted.")
    
    def extract_normalized_numeric_data(self, method: str = 'standard'):
        """Return normalized numeric data."""
        num_df = self.df.select_dtypes(include=[np.number]).copy()
        if num_df.empty:
            return pd.DataFrame()
        
        scalers = {
            'minmax': MinMaxScaler(),
            'standard': StandardScaler(),
            'robust': RobustScaler()
        }
        scaler = scalers.get(method, StandardScaler())
        
        scaled = scaler.fit_transform(num_df)
        return pd.DataFrame(scaled, columns=num_df.columns, index=num_df.index)
    
    def extract_normalized_categorical_data(self, method: str = 'onehot'):
        """Return encoded categorical data."""
        cat_df = self.df.select_dtypes(include=['object', 'category']).copy()
        if cat_df.empty:
            return pd.DataFrame()
        
        if method == 'onehot':
            encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
            encoded = encoder.fit_transform(cat_df)
            cols = encoder.get_feature_names_out(cat_df.columns)
            return pd.DataFrame(encoded, columns=cols, index=cat_df.index)
        
        elif method == 'ordinal':
            encoder = OrdinalEncoder()
            encoded = encoder.fit_transform(cat_df)
            return pd.DataFrame(encoded, columns=cat_df.columns, index=cat_df.index)
        
        elif method == 'uniform':
            encoded = pd.DataFrame()
            for col in cat_df.columns:
                ranks = cat_df[col].rank(method='dense') - 1
                encoded[col] = ranks / ranks.max() if ranks.max() > 0 else ranks
            return encoded
        
        return pd.DataFrame()
    
    def get_prepared_dataset(self, num_method='standard', cat_method='onehot'):
        """Return unified DataFrame with scaled numeric + encoded categorical data."""
        num_scaled = self.extract_normalized_numeric_data(num_method)
        cat_encoded = self.extract_normalized_categorical_data(cat_method)
        
        if num_scaled.empty and cat_encoded.empty:
            return self.df
        elif num_scaled.empty:
            return cat_encoded
        elif cat_encoded.empty:
            return num_scaled
        else:
            return pd.concat([num_scaled, cat_encoded], axis=1)
    
    def plot_univariate(self, column: str):
        """3-panel subplot for numeric columns."""
        if column not in self.df.columns or not pd.api.types.is_numeric_dtype(self.df[column]):
            print("Column is not numeric or doesn't exist.")
            return
        
        data = self.df[column].dropna()
        
        fig = make_subplots(
            rows=1, cols=3,
            subplot_titles=("Violin + Box", "Index vs Value", "Histogram"),
            specs=[[{"type": "violin"}, {"type": "scatter"}, {"type": "histogram"}]]
        )
        
        fig.add_trace(go.Violin(y=data, name="Violin", box_visible=True, meanline_visible=True), row=1, col=1)
        fig.add_trace(go.Scatter(x=data.index, y=data, mode='markers', name="Points"), row=1, col=2)
        fig.add_trace(go.Histogram(x=data, nbinsx=30, name="Histogram"), row=1, col=3)
        
        fig.update_layout(title=f"Univariate Analysis: {column}", height=500)
        fig.show()
    
    def plot_relationship(self, col1: str, col2: str):
        """Smart relationship plot based on column types."""
        if col1 not in self.df.columns or col2 not in self.df.columns:
            print("One or both columns not found.")
            return
        
        is_num1 = pd.api.types.is_numeric_dtype(self.df[col1])
        is_num2 = pd.api.types.is_numeric_dtype(self.df[col2])
        
        if is_num1 and is_num2:
            fig = px.scatter(self.df, x=col1, y=col2, trendline="ols", title=f"{col1} vs {col2}")
        elif (is_num1 and not is_num2) or (not is_num1 and is_num2):
            cat_col = col2 if not is_num2 else col1
            num_col = col1 if is_num1 else col2
            fig = px.box(self.df, x=cat_col, y=num_col, points="all", title=f"{col1} vs {col2}")
        else:
            fig = px.histogram(self.df, x=col1, color=col2, barmode='group', title=f"{col1} vs {col2}")
        
        fig.show()
    
    def plot_categorical_frequency(self, column: str):
        """Display bar chart with percentages."""
        if column not in self.df.columns:
            print("Column not found.")
            return
        html = self.plotter.plot_bar(self.df[column], f"Frequency of {column}")
        display(HTML(html))
    
    def plot_all_associations_heatmap(self):
        """Numeric correlation heatmap (Pearson)."""
        num_df = self.df.select_dtypes(include=[np.number])
        if len(num_df.columns) < 2:
            print("Not enough numeric columns.")
            return
        
        corr = num_df.corr()
        fig = px.imshow(corr, text_auto=True, aspect="auto",
                       color_continuous_scale='RdBu_r',
                       title="Correlation Heatmap (Pearson r)")
        fig.show()
    
    def run_full_demo(self):
        """Run complete demonstration flow."""
        print("🚀 Starting Full Demo Flow...")
        self.get_summary()
        self.handle_missing_values(strategy='median')
        self.remove_duplicates()
        self.handle_outliers(method='cap')
        print("\n📊 Prepared Dataset (Normalized + Encoded):")
        display(self.get_prepared_dataset().head())

## 4. Usage Example (Titanic Dataset)

In [ ]:
# Initialize
inspector = DataInspector()

# 1. Upload Data
inspector.upload_data()

# 2. Explore
inspector.get_summary()

# 3. Clean
inspector.handle_missing_values(strategy='median')
inspector.remove_duplicates()
inspector.handle_outliers(method='cap')

# 4. Feature Engineering
prepared = inspector.get_prepared_dataset(num_method='standard', cat_method='onehot')
print("Prepared shape:", prepared.shape)

# 5. Visualizations
if 'age' in inspector.df.columns:
    inspector.plot_univariate('age')

if 'pclass' in inspector.df.columns and 'fare' in inspector.df.columns:
    inspector.plot_relationship('pclass', 'fare')

inspector.plot_all_associations_heatmap()